In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('.'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

./creditcard.csv
./.config/.last_update_check.json
./.config/.last_opt_in_prompt.yaml
./.config/default_configs.db
./.config/active_config
./.config/gce
./.config/.last_survey_prompt.yaml
./.config/config_sentinel
./.config/hidden_gcloud_config_universe_descriptor_data_cache_configs.db
./.config/logs/2025.12.09/14.41.33.792924.log
./.config/logs/2025.12.09/14.41.43.412452.log
./.config/logs/2025.12.09/14.41.42.675750.log
./.config/logs/2025.12.09/14.41.27.893750.log
./.config/logs/2025.12.09/14.41.18.717681.log
./.config/logs/2025.12.09/14.40.47.605300.log
./.config/configurations/config_default
./sample_data/README.md
./sample_data/anscombe.json
./sample_data/california_housing_train.csv
./sample_data/mnist_test.csv
./sample_data/california_housing_test.csv
./sample_data/mnist_train_small.csv


In [2]:
!pip install -U pip
!git clone https://github.com/riri324/CTGAN.git
%cd CTGAN
!pip install -e .

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 83.8 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
Cloning into 'CTGAN'...
remote: Enumerating objects: 2361, done.
remote: Counting objects: 100% (1013/1013), done.
remote: Compressing objects: 100% (261/261), done.
remote: Total 2361 (delta 926), reused 753 (delta 752), pack-reused 1348 (from 3)
Receiving objects: 100% (2361/2361), 1.97 MiB | 5.34 MiB/s, done.
Resolving deltas: 100% (1473/1473), done.
/content/CTGAN
Obtaining file:///content/CTGAN
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 55.1 MB/s  0:00:00
  Building editable for ctgan (pyproject.toml) ... done
  Created wheel for ctgan: filename=ctg

In [3]:
import pkgutil, sys
print("python:", sys.executable)
print("ctgan exists?", any(m.name == "ctgan" for m in pkgutil.iter_modules()))


python: /usr/bin/python3
ctgan exists? True


In [4]:
import os
import pandas as pd

data_path = "/content/creditcard.csv"

df = pd.read_csv(data_path)

print("Dataset Shape:", df.shape)
df.head()


Dataset Shape: (284807, 31)


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [5]:
# USE_SMALL = False

In [6]:
# from sklearn.model_selection import train_test_split

# # use 5% of datasets
# if USE_SMALL:
#     df, _ = train_test_split(
#         df,
#         test_size=0.95,
#         stratify=df["Class"],
#         random_state=42
#     )
#     print("Using SMALL dataset:", df.shape)


In [7]:
import numpy as np
import pandas as pd
import time

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, recall_score

from xgboost import XGBClassifier
from ctgan import CTGAN

# Hold-out split: Baseline / GAN / Diffusion for same standards for test
train_df, test_df = train_test_split(
    df, test_size=0.2, stratify=df["Class"], random_state=42
)

train_base, val_df = train_test_split(
    train_df, test_size=0.2, stratify=train_df["Class"], random_state=42
)
FEATURE_COLS = [c for c in df.columns if c != "Class"]

# train_df, test_df = train_test_split(
#     df_small, test_size=0.2, stratify=df_small["Class"], random_state=42
# )

# train_base, val_df = train_test_split(
#     train_df, test_size=0.2, stratify=train_df["Class"], random_state=42
# )

# FEATURE_COLS = [c for c in df_small.columns if c != "Class"]

print("Train fraud rate:", train_df["Class"].mean())
print("Test  fraud rate:", test_df["Class"].mean())

# ratios = [0.3, 0.5, 0.7, 1.0]
# ratios_ext = [None] + ratios          # natural (None)
discrete_columns = ["Class"]

# def ratio_name(r):
#     return "natural" if r is None else f"forced_{r:.2f}"

scale_pos_weight = (train_df["Class"].value_counts()[0] / train_df["Class"].value_counts()[1])
print("scale_pos_weight:", scale_pos_weight)

Train fraud rate: 0.001729245759178389
Test  fraud rate: 0.0017204452090867595
scale_pos_weight: 577.2868020304569


In [8]:
# def find_best_threshold(prob, y_true):
#     thresholds = np.linspace(0.001, 0.5, 200)
#     best_t, best_f1 = 0.5, 0.0
#     for t in thresholds:
#         pred = (prob > t).astype(int)
#         f1 = f1_score(y_true, pred)
#         if f1 > best_f1:
#             best_f1, best_t = float(f1), float(t)
#     return best_t, best_f1

In [9]:
from sklearn.metrics import pairwise_distances

def privacy_nn_memorization(real_df, syn_df, feature_cols, threshold=1e-6,
                            sample_real=5000, sample_syn=5000, seed=42):
    real_s = real_df.sample(n=min(sample_real, len(real_df)), random_state=seed)
    syn_s  = syn_df.sample(n=min(sample_syn, len(syn_df)), random_state=seed)

    real_X = real_s[feature_cols].values
    syn_X  = syn_s[feature_cols].values

    dists = pairwise_distances(syn_X, real_X, metric="euclidean")
    min_dists = dists.min(axis=1)

    out = {
        "NN_mean": float(min_dists.mean()),
        "NN_min": float(min_dists.min()),
        "NearDup_rate(thr)": float((min_dists < threshold).mean())
    }

    real_fraud = real_df[real_df.Class==1]
    syn_fraud  = syn_df[syn_df.Class==1]
    if len(real_fraud) > 0 and len(syn_fraud) > 0:
        rf = real_fraud.sample(n=min(2000, len(real_fraud)), random_state=seed)[feature_cols].values
        sf = syn_fraud.sample(n=min(2000, len(syn_fraud)), random_state=seed)[feature_cols].values
        d_f = pairwise_distances(sf, rf, metric="euclidean")
        min_d_f = d_f.min(axis=1)
        out["Fraud_NN_mean"] = float(min_d_f.mean())
        out["Fraud_NN_min"]  = float(min_d_f.min())
    else:
        out["Fraud_NN_mean"] = np.nan
        out["Fraud_NN_min"]  = np.nan

    return out

def real_real_nn_baseline(real_df, feature_cols, class_value=None, n_sample=2000, seed=42):
    if class_value is None:
        X = real_df[feature_cols].values
    else:
        X = real_df[real_df["Class"] == class_value][feature_cols].values

    rng = np.random.default_rng(seed)
    n = min(n_sample, len(X))
    if n < 2:
        return {"RealReal_NN_mean": np.nan, "RealReal_NN_min": np.nan, "n_used": n}

    idx1 = rng.choice(len(X), n, replace=False)
    idx2 = rng.choice(len(X), n, replace=False)
    X1, X2 = X[idx1], X[idx2]

    d = pairwise_distances(X1, X2, metric="euclidean")
    np.fill_diagonal(d, np.inf)
    min_d = d.min(axis=1)
    return {"RealReal_NN_mean": float(min_d.mean()), "RealReal_NN_min": float(min_d.min()), "n_used": n}


In [10]:
privacy_rows_ctgan = []
imbalance_rows_ctgan = []

# Real-Real baseline
baseline_all  = real_real_nn_baseline(train_base, FEATURE_COLS, class_value=None, n_sample=2000)
baseline_fraud = real_real_nn_baseline(train_base, FEATURE_COLS, class_value=1, n_sample=2000)

baseline_df = pd.DataFrame([
    {"syn_ratio": "BASELINE_REALREAL_ALL",
     "NN_mean": baseline_all["RealReal_NN_mean"],
     "NN_min":  baseline_all["RealReal_NN_min"]},
    {"syn_ratio": "BASELINE_REALREAL_FRAUD",
     "Fraud_NN_mean": baseline_fraud["RealReal_NN_mean"],
     "Fraud_NN_min":  baseline_fraud["RealReal_NN_min"]},
]).set_index("syn_ratio")


In [11]:
from scipy.stats import ks_2samp, wasserstein_distance

def distribution_similarity(real_df, syn_df, exclude_cols=["Class"]):
    ks_scores, wass_scores = [], []
    for col in real_df.columns:
        if col not in exclude_cols:
            ks_scores.append(ks_2samp(real_df[col], syn_df[col]).statistic)
            wass_scores.append(wasserstein_distance(real_df[col], syn_df[col]))
    return {
        "KS_mean": float(np.mean(ks_scores)),
        "Wasserstein_mean": float(np.mean(wass_scores))
    }

def fraud_collapse_check(syn_df, threshold=0.001):
    fr = float(syn_df["Class"].mean())
    return {"Fraud_Ratio": fr, "Collapsed": fr < threshold}

def loss_variance(x):
    return float(np.var(x))


In [12]:
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, recall_score

def evaluate_holdout_xgb_simple(train_any_df, test_df, xgb_params, threshold=0.5):

    X_train = train_any_df.drop(columns=["Class"])
    y_train = train_any_df["Class"]

    X_test = test_df.drop(columns=["Class"])
    y_test = test_df["Class"]

    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)

    model = XGBClassifier(**xgb_params)
    model.fit(X_train_s, y_train)

    prob = model.predict_proba(X_test_s)[:, 1]
    pred = (prob > threshold).astype(int)

    return {
        "ROC-AUC": roc_auc_score(y_test, prob),
        "AUPRC": average_precision_score(y_test, prob),
        "F1": f1_score(y_test, pred),
        "Recall": recall_score(y_test, pred)
    }


In [13]:
def make_hybrid(real_df, syn_df):
    return pd.concat([real_df, syn_df], axis=0)\
             .sample(frac=1, random_state=42)\
             .reset_index(drop=True)


In [14]:
# Hold-out Baseline / GAN / Diffusion same parameters for fairness
XGB_PARAMS = dict(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="aucpr",
    random_state=42,
    n_jobs=-1
)
XGB_PARAMS_ON  = {**XGB_PARAMS, "scale_pos_weight": scale_pos_weight}
XGB_PARAMS_OFF = {**XGB_PARAMS}

# baseline_holdout = evaluate_holdout_xgb(train_df, test_df, XGB_PARAMS)
# baseline_holdout

In [15]:
CTGAN_BASE = dict(
    epochs=300,
    #epochs=50,
    batch_size=500,
    verbose=False,
    ratio=None,              # natural
    embedding_dim=128,
    generator_dim=(256, 256),
    discriminator_dim=(256, 256),
    discriminator_steps=1,
    log_frequency=True,
    pac=10,
    enable_gpu=True
)


In [16]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [17]:
def run_ctgan_sweep(param_name, param_values, base_kwargs,
                    train_base, test_df, discrete_columns,
                    feature_cols,
                    do_privacy=True, privacy_only_values=None):

    ctgan_models = {}
    syn_by_setting = {}
    results = {}
    quality = {}
    privacy_rows = []

    for v in param_values:
        setting_name = f"{param_name}={v}"
        print(f"\n[CTGAN sweep] {setting_name}")

        kwargs = dict(base_kwargs)
        kwargs[param_name] = v

        ctgan = CTGAN(**kwargs)
        ctgan.set_random_state(42)

        ctgan.fit(train_base, discrete_columns)
        syn = ctgan.sample(len(train_base))

        ctgan_models[setting_name] = ctgan
        syn_by_setting[setting_name] = syn

        # ---- Privacy ----
        if do_privacy:
            run_priv = True
            if privacy_only_values is not None:
                run_priv = (v in privacy_only_values)
            if run_priv:
                priv = privacy_nn_memorization(
                    real_df=train_base,
                    syn_df=syn,
                    feature_cols=feature_cols,
                    threshold=1e-6,
                    sample_real=5000,
                    sample_syn=5000,
                    seed=42
                )
                privacy_rows.append({"setting": setting_name, **priv})

        # ---- ROC-AUC / AUPRC ----
        uniq = np.unique(syn["Class"].values)
        if len(uniq) >= 2:
            results[(setting_name, "syn_only_spw_ON")]  = evaluate_holdout_xgb_simple(syn, test_df, XGB_PARAMS_ON)
            results[(setting_name, "syn_only_spw_OFF")] = evaluate_holdout_xgb_simple(syn, test_df, XGB_PARAMS_OFF)
        else:
            print(f"  -> skip syn-only (only one class in syn): {uniq}")

        # ---- hybrid ----
        hybrid = make_hybrid(train_base, syn)
        results[(setting_name, "hybrid_spw_ON")]  = evaluate_holdout_xgb_simple(hybrid, test_df, XGB_PARAMS_ON)
        results[(setting_name, "hybrid_spw_OFF")] = evaluate_holdout_xgb_simple(hybrid, test_df, XGB_PARAMS_OFF)

        # ---- Quality: KS/Wass + collapse + loss var ----
        real_sample = train_base.sample(
            n=min(len(train_base), len(syn)),
            replace=False,
            random_state=42
        )
        syn_sample = syn.sample(n=len(real_sample), random_state=42)

        lv = ctgan.loss_values
        gen_var  = loss_variance(lv["Generator Loss"].values) if "Generator Loss" in lv else np.nan
        disc_var = loss_variance(lv["Discriminator Loss"].values) if "Discriminator Loss" in lv else np.nan

        quality[setting_name] = {
            **distribution_similarity(real_sample, syn_sample),
            **fraud_collapse_check(syn),
            "Gen_Loss_Var": gen_var,
            "Disc_Loss_Var": disc_var
        }

    perf_df = pd.DataFrame(results).T
    perf_df.index = pd.MultiIndex.from_tuples(perf_df.index, names=[param_name, "train_setting"])
    perf_df = perf_df.sort_index()

    quality_df = pd.DataFrame(quality).T.sort_index()

    privacy_df = (
        pd.DataFrame(privacy_rows).set_index("setting")
        if len(privacy_rows) else pd.DataFrame()
    )

    return perf_df, quality_df, privacy_df, syn_by_setting, ctgan_models


In [18]:
epochs_list = [100, 200]

perf_epochs, qual_epochs, priv_epochs, syn_epochs, models_epochs = run_ctgan_sweep(
    param_name="epochs",
    param_values=epochs_list,
    base_kwargs=CTGAN_BASE,
    train_base=train_base,
    test_df=test_df,
    discrete_columns=discrete_columns,
    feature_cols=FEATURE_COLS,
    do_privacy=True,
    privacy_only_values=[100, 200]
)

display(perf_epochs)
display(qual_epochs)
display(priv_epochs)
perf_epochs.to_csv("ablation_epochs_performance.csv")
qual_epochs.to_csv("ablation_epochs_quality.csv")
priv_epochs.to_csv("ablation_epochs_privacy.csv")



[CTGAN sweep] epochs=100


/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:841: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:270.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass



[CTGAN sweep] epochs=200


ROC-AUC     AUPRC        F1    Recall
epochs     train_setting                                           
epochs=100 hybrid_spw_OFF    0.977271  0.810740  0.765550  0.816327
           hybrid_spw_ON     0.971243  0.803233  0.188804  0.877551
           syn_only_spw_OFF  0.970081  0.707505  0.115903  0.877551
           syn_only_spw_ON   0.964146  0.701886  0.036580  0.908163
epochs=200 hybrid_spw_OFF    0.976371  0.826464  0.742081  0.836735
           hybrid_spw_ON     0.981262  0.814650  0.088478  0.928571
           syn_only_spw_OFF  0.976701  0.650485  0.146502  0.908163
           syn_only_spw_ON   0.970694  0.741728  0.019587  0.948980

,KS_mean,Wasserstein_mean,Fraud_Ratio,Collapsed,Gen_Loss_Var,Disc_Loss_Var
epochs=100,0.141848,295.570145,0.321875,False,2.886281,0.044758
epochs=200,0.13429,156.469544,0.322588,False,1.250957,0.024566


,NN_mean,NN_min,NearDup_rate(thr),Fraud_NN_mean,Fraud_NN_min
setting,,,,,
epochs=100,77.624643,3.397525,0.0,542.596644,8.475639
epochs=200,79.523858,4.271436,0.0,546.326589,8.471281


In [19]:
batch_list = [250, 500, 750]

perf_batch, qual_batch, priv_batch, syn_batch, models_batch = run_ctgan_sweep(
    param_name="batch_size",
    param_values=batch_list,
    base_kwargs=CTGAN_BASE,
    train_base=train_base,
    test_df=test_df,
    discrete_columns=discrete_columns,
    feature_cols=FEATURE_COLS,
    do_privacy=True,
    privacy_only_values=[250, 750]
)

display(perf_batch)
display(qual_batch)
display(priv_batch)
perf_batch.to_csv("ablation_batch_performance.csv")
qual_batch.to_csv("ablation_batch_quality.csv")
priv_batch.to_csv("ablation_batch_privacy.csv")



[CTGAN sweep] batch_size=250


KeyboardInterrupt: 

In [20]:
def run_ctgan_dim_sweep(dim_list):
    results = {}
    quality = {}
    privacy_rows = []

    for d in dim_list:
        setting = f"dim={d}"
        print(f"\n[CTGAN sweep] {setting}")

        kwargs = dict(CTGAN_BASE)
        kwargs["generator_dim"] = d
        kwargs["discriminator_dim"] = d

        ctgan = CTGAN(**kwargs)
        ctgan.set_random_state(42)
        ctgan.fit(train_base, discrete_columns)
        syn = ctgan.sample(len(train_base))

        # performance
        uniq = np.unique(syn["Class"].values)
        if len(uniq) >= 2:
            results[(setting, "syn_only_spw_ON")]  = evaluate_holdout_xgb_simple(syn, test_df, XGB_PARAMS_ON)
            results[(setting, "syn_only_spw_OFF")] = evaluate_holdout_xgb_simple(syn, test_df, XGB_PARAMS_OFF)

        hybrid = make_hybrid(train_base, syn)
        results[(setting, "hybrid_spw_ON")]  = evaluate_holdout_xgb_simple(hybrid, test_df, XGB_PARAMS_ON)
        results[(setting, "hybrid_spw_OFF")] = evaluate_holdout_xgb_simple(hybrid, test_df, XGB_PARAMS_OFF)

        # privacy
        if d == (128, 128):
            priv = privacy_nn_memorization(train_base, syn, FEATURE_COLS)
            privacy_rows.append({"setting": setting, **priv})
        elif d == (512, 512):
            priv = privacy_nn_memorization(train_base, syn, FEATURE_COLS)
            privacy_rows.append({"setting": setting, **priv})

        # quality
        real_sample = train_base.sample(n=min(len(train_base), len(syn)), replace=False, random_state=42)
        syn_sample  = syn.sample(n=len(real_sample), random_state=42)
        quality[setting] = {**distribution_similarity(real_sample, syn_sample), **fraud_collapse_check(syn)}

    perf_df = pd.DataFrame(results).T
    perf_df.index = pd.MultiIndex.from_tuples(perf_df.index, names=["dim", "train_setting"])
    perf_df = perf_df.sort_index()

    qual_df = pd.DataFrame(quality).T.sort_index()
    priv_df = pd.DataFrame(privacy_rows).set_index("setting") if len(privacy_rows) else pd.DataFrame()
    return perf_df, qual_df, priv_df

dim_list = [(128,128), (512,512)]
perf_dim, qual_dim, priv_dim = run_ctgan_dim_sweep(dim_list)

display(perf_dim)
display(qual_dim)
display(priv_dim)
perf_dim.to_csv("ablation_dim_performance.csv")
qual_dim.to_csv("ablation_dim_quality.csv")
priv_dim.to_csv("ablation_dim_privacy.csv")





[CTGAN sweep] dim=(128, 128)

[CTGAN sweep] dim=(512, 512)


ROC-AUC     AUPRC        F1    Recall
dim            train_setting                                           
dim=(128, 128) hybrid_spw_OFF    0.972822  0.826214  0.764977  0.846939
               hybrid_spw_ON     0.975606  0.807006  0.065489  0.908163
               syn_only_spw_OFF  0.971989  0.686294  0.138148  0.897959
               syn_only_spw_ON   0.968480  0.706131  0.014447  0.948980
dim=(512, 512) hybrid_spw_OFF    0.976801  0.845954  0.724891  0.846939
               hybrid_spw_ON     0.979889  0.826712  0.183505  0.908163
               syn_only_spw_OFF  0.974567  0.660733  0.149660  0.897959
               syn_only_spw_ON   0.976453  0.639122  0.053293  0.908163

,KS_mean,Wasserstein_mean,Fraud_Ratio,Collapsed
"dim=(128, 128)",0.138561,242.924168,0.324305,False
"dim=(512, 512)",0.127816,120.416022,0.324316,False


,NN_mean,NN_min,NearDup_rate(thr),Fraud_NN_mean,Fraud_NN_min
setting,,,,,
"dim=(128, 128)",91.183164,3.724330,0.0,545.863996,6.672483
"dim=(512, 512)",82.062869,3.927644,0.0,569.428956,14.584265


In [22]:
log_list = [False]

perf_log, qual_log, priv_log, syn_log, models_log = run_ctgan_sweep(
    param_name="log_frequency",
    param_values=log_list,
    base_kwargs=CTGAN_BASE,
    train_base=train_base,
    test_df=test_df,
    discrete_columns=discrete_columns,
    feature_cols=FEATURE_COLS,
    do_privacy=True,
    privacy_only_values=[False]
)

display(perf_log)
display(qual_log)
display(priv_log)
perf_log.to_csv("ablation_log_performance.csv")
qual_log.to_csv("ablation_log_quality.csv")
priv_log.to_csv("ablation_log_privacy.csv")




[CTGAN sweep] log_frequency=False


ROC-AUC     AUPRC        F1    Recall
log_frequency       train_setting                                           
log_frequency=False hybrid_spw_OFF    0.977054  0.863922  0.874317  0.816327
                    hybrid_spw_ON     0.976042  0.863815  0.861538  0.857143
                    syn_only_spw_OFF  0.906626  0.381358  0.059406  0.030612
                    syn_only_spw_ON   0.911930  0.243290  0.020000  0.010204

,KS_mean,Wasserstein_mean,Fraud_Ratio,Collapsed,Gen_Loss_Var,Disc_Loss_Var
log_frequency=False,0.071192,152.331931,0.001662,False,2.051419,0.045616


,NN_mean,NN_min,NearDup_rate(thr),Fraud_NN_mean,Fraud_NN_min
setting,,,,,
log_frequency=False,66.652988,3.249699,0.0,780.603641,14.169337


In [ ]:
dsteps_list = [3, 5]

perf_dsteps, qual_dsteps, priv_dsteps, syn_dsteps, models_dsteps = run_ctgan_sweep(
    param_name="discriminator_steps",
    param_values=dsteps_list,
    base_kwargs=CTGAN_BASE,
    train_base=train_base,
    test_df=test_df,
    discrete_columns=discrete_columns,
    feature_cols=FEATURE_COLS,
    do_privacy=True,
    privacy_only_values=[3,5]
)

display(perf_dsteps)
display(qual_dsteps)
display(priv_dsteps)
perf_dsteps.to_csv("ablation_dsteps_performance.csv")
qual_dsteps.to_csv("ablation_dsteps_quality.csv")
priv_dsteps.to_csv("ablation_dsteps_privacy.csv")



[CTGAN sweep] discriminator_steps=3

[CTGAN sweep] discriminator_steps=5


In [ ]:
#print(ctgan_models["natural"].loss_values.columns)

In [ ]:
# import matplotlib.pyplot as plt

# imbalance_ctgan_df = pd.DataFrame(imbalance_rows_ctgan)

# imbalance_ctgan_df["requested_r"] = imbalance_ctgan_df["syn_ratio_setting"].apply(
#     lambda s: 0.0 if s == "natural" else float(s.replace("forced_", ""))
# )
# imbalance_ctgan_df = imbalance_ctgan_df.sort_values("requested_r")

# plt.figure(figsize=(6,4), dpi=150)
# plt.plot(imbalance_ctgan_df["requested_r"], imbalance_ctgan_df["hybrid_fraud_ratio"], marker="o")
# plt.xlabel("Requested fraud ratio (r)  (natural shown as 0)")
# plt.ylabel("Actual fraud ratio in hybrid")
# plt.title("Imbalance mitigation via CTGAN synthetic data")
# plt.grid(True, linestyle="dotted")
# plt.tight_layout()
# plt.show()

# display(imbalance_ctgan_df)


In [ ]:
# from sklearn.manifold import TSNE
# import matplotlib.pyplot as plt
# import numpy as np
# from sklearn.preprocessing import StandardScaler
# import pandas as pd

# def tsne_real_vs_syn(
#     real_df,
#     syn_df,
#     label_col="Class",
#     sample_per_group=5000,
#     perplexity=30,
#     random_state=42,
#     point_size=10,
#     alpha=0.55
# ):

#     real_X = real_df.drop(columns=[label_col]).copy()
#     syn_X  = syn_df.drop(columns=[label_col]).copy()

#     real_X = real_X.sample(n=min(sample_per_group, len(real_X)), random_state=random_state)
#     syn_X  = syn_X.sample(n=min(sample_per_group, len(syn_X)), random_state=random_state)

#     X = pd.concat([real_X, syn_X], axis=0).reset_index(drop=True)
#     domain = np.array(["Real"] * len(real_X) + ["Synthetic"] * len(syn_X))

#     scaler = StandardScaler()
#     X_scaled = scaler.fit_transform(X)

#     # t-SNE
#     tsne = TSNE(
#         n_components=2,
#         perplexity=perplexity,
#         init="pca",
#         learning_rate="auto",
#         random_state=random_state
#     )
#     Z = tsne.fit_transform(X_scaled)

#     # Plot
#     plt.style.use("seaborn-v0_8")

#     fig, ax = plt.subplots(figsize=(7, 6), dpi=150)

#     ax.set_facecolor("#EAEAF2")
#     ax.grid(True, color="white", linewidth=1.2)
#     ax.set_axisbelow(True)

#     for spine in ax.spines.values():
#         spine.set_visible(False)

#     for name in ["Real", "Synthetic"]:
#         idx = (domain == name)
#         ax.scatter(
#             Z[idx, 0], Z[idx, 1],
#             s=point_size,
#             alpha=alpha,
#             edgecolors="none",
#             label=name
#         )

#     ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

#     ax.legend(frameon=True, framealpha=0.9)

#     plt.tight_layout()
#     plt.show()


In [ ]:
# def tsne_fraud_only(real_df, syn_df, label_col="Class", sample=2000):
#     real_fraud = real_df[real_df[label_col] == 1]
#     syn_fraud  = syn_df[syn_df[label_col] == 1]

#     if len(real_fraud) == 0 or len(syn_fraud) == 0:
#         print("Fraud samples missing in real or synthetic. Check generation ratio.")
#         return

#     tsne_real_vs_syn(
#         real_fraud,
#         syn_fraud,
#         label_col=label_col,
#         sample_per_group=sample,
#         perplexity=20
#     )

# tsne_targets = ["natural", "forced_0.50"]

# for name in tsne_targets:
#     print(f"t-SNE visualization for {name}")
#     tsne_real_vs_syn(train_base, ctgan_syn[name])

# for name in tsne_targets:
#     print(f"t-SNE FraudOnly visualization for {name}")
#     tsne_fraud_only(train_base, ctgan_syn[name])

In [ ]:
# # Synthetic Quality Metrics

# from scipy.stats import ks_2samp, wasserstein_distance

# def distribution_similarity(real_df, syn_df, exclude_cols=["Class"]):
#     ks_scores, wass_scores = [], []
#     for col in real_df.columns:
#         if col not in exclude_cols:
#             ks_scores.append(ks_2samp(real_df[col], syn_df[col]).statistic)
#             wass_scores.append(wasserstein_distance(real_df[col], syn_df[col]))
#     return {
#         "KS_mean": float(np.mean(ks_scores)),
#         "Wasserstein_mean": float(np.mean(wass_scores))
#     }

# def fraud_collapse_check(syn_df, threshold=0.001):
#     fr = float(syn_df["Class"].mean())
#     return {"Fraud_Ratio": fr, "Collapsed": fr < threshold}

# def loss_variance(x):
#     return float(np.var(x))

# ctgan_quality = {}

# for name, syn in ctgan_syn.items():
#     real_sample = train_base.sample(
#         n=min(len(train_base), len(syn)),
#         replace=False,
#         random_state=42
#     )
#     syn_sample = syn.sample(n=len(real_sample), random_state=42)

#     lv = ctgan_models[name].loss_values
#     gen_var = loss_variance(lv["Generator Loss"].values) if "Generator Loss" in lv else np.nan
#     disc_var = loss_variance(lv["Discriminator Loss"].values) if "Discriminator Loss" in lv else np.nan

#     ctgan_quality[name] = {
#         **distribution_similarity(real_sample, syn_sample),
#         **fraud_collapse_check(syn),
#         "Gen_Loss_Var": gen_var,
#         "Disc_Loss_Var": disc_var,
#     }

# ctgan_quality_df = pd.DataFrame(ctgan_quality).T.sort_index()
# ctgan_quality_df.to_csv("ctgan_quality_with_natural.csv")
# display(ctgan_quality_df)
